In [ ]:
import os
import wandb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import numpy as np
import pandas as pd
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
import torchvision.transforms as transforms

try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

from PIL import Image

import matplotlib.pyplot as plt
%matplotlib inline

[INFO] Couldn't find torchinfo... installing it.


### Зафиксируем seed для воспроизводимости

In [ ]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

### Задаем параметры (конфиг)

In [ ]:
all_brands = ('Audi', 'BMW', 'Chevrolet', 'Hyundai', 'Kia', 'Lexus', 'Mercedes-Benz', 'Toyota')
all_body_types = ('sedan', 'SUV')

class CFG:
  api = ""
  project = "kolesa-cars"
  entity = "armntvs-d3v-student"
  num_epochs = 10
  train_batch_size = 32
  test_batch_size = 512
  num_workers = 0
  lr = 3e-4
  seed = 42
  classes = all_body_types
  wandb = True

In [ ]:
# вход в W&B (ключ спросит один раз, берём с https://wandb.ai/authorize)
wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: armntvs-d3v (armntvs-d3v-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
seed_everything(CFG.seed)

In [ ]:
# Переведем наш класс с параметрами в словарь

def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

### Импортируем наш датасет с картинками

In [ ]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  !cp -r /content/drive/MyDrive/gp5_project/data /content/data

  base_dir = '/content/data'
except:
  base_dir = '../data'

from pathlib import Path

base_dir = Path(base_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_bt = pd.read_csv(base_dir / 'prepared_df_images.csv')
df_bt

,kolesa_id,body_type,img_filename,target
0,220599328,Седан,220599328.webp,0
1,222486528,Седан,222486528.webp,0
2,218695550,Внедорожник,218695550.webp,1
3,214781208,Седан,214781208.webp,0
4,222346737,Седан,222346737.webp,0
...,...,...,...,...
4428,222478046,Седан,222478046.webp,0
4429,222704181,Седан,222704181.webp,0
4430,220000872,Седан,220000872.webp,0
4431,222542135,Седан,222542135.webp,0


In [ ]:
df_brand = pd.read_csv(base_dir / 'brand_df_images.csv')
df_brand

,kolesa_id,brand,body_type,img_filename,target
0,220599328,Chevrolet,Седан,220599328.webp,2
1,222486528,Chevrolet,Седан,222486528.webp,2
2,218695550,Chevrolet,Внедорожник,218695550.webp,2
3,214781208,Chevrolet,Седан,214781208.webp,2
4,222346737,Chevrolet,Седан,222346737.webp,2
...,...,...,...,...,...
4428,222478046,Hyundai,Седан,222478046.webp,3
4429,222704181,Hyundai,Седан,222704181.webp,3
4430,220000872,Hyundai,Седан,220000872.webp,3
4431,222542135,Hyundai,Седан,222542135.webp,3


In [ ]:
train_df_bt, test_all_df_bt = train_test_split(df_bt, test_size=0.3, stratify=df_bt['target'], random_state=CFG.seed)
val_df_bt, test_df_bt = train_test_split(test_all_df_bt, test_size=0.5, stratify=test_all_df_bt['target'], random_state=CFG.seed)

train_df_brand, test_all_df_brand = train_test_split(df_brand, test_size=0.3, stratify=df_brand['target'], random_state=CFG.seed)
val_df_brand, test_df_brand = train_test_split(test_all_df_brand, test_size=0.5, stratify=test_all_df_brand['target'], random_state=CFG.seed)

In [ ]:
# https://stackoverflow.com/questions/65231299/load-csv-and-image-dataset-in-pytorch

class KolesaDataset(torch.utils.data.Dataset):
    def __init__(self, df, images_folder = base_dir / 'images', transform = None):
        # т.к. после train_test_split индексы сохряняются из исходного датасета, то мы их сбрасываем и удаляем колонку со старыми индексами для корректной работы __getitem___
        self.df = df.reset_index().drop(columns=['index'])
        self.images_folder = images_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        filename = self.df.loc[index, 'img_filename']
        label = self.df.loc[index, 'target']
        image = Image.open(os.path.join(self.images_folder, filename)).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, label



### Функции обучения и инференса

In [ ]:
# функция обучения
def train(model, device, train_loader, optimizer, criterion, epoch, WANDB):
    model.train()
    train_loss = 0
    correct = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # обнуляем градиенты
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        train_loss = criterion(output, target) # считаем лосс
        train_loss.backward() # обратный проход
        optimizer.step() # делаем шаг оптимизатором

    tqdm.write('\nTrain set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        train_loss, 100. * correct / len(train_loader.dataset)))

    # логируем функцию потерь и точность
    if WANDB:
        wandb.log({'train_loss': train_loss,
                   'train_accuracy': correct / len(train_loader.dataset)})

In [ ]:
# функция инференса
def test(model, device, test_loader, criterion, WANDB):
    model.eval()
    test_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            # https://stackoverflow.com/questions/53467215/convert-pytorch-cuda-tensor-to-numpy-array
            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
    report = classification_report(targets, preds, target_names=CFG.classes)

    tqdm.write('Test set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        test_loss, 100. * correct / len(test_loader.dataset)))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'test_loss': test_loss,
                   'test_accuracy': correct / len(test_loader.dataset)})

# функция инференса
def val(model, device, val_loader, criterion, WANDB):
    model.eval()
    val_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    report = classification_report(targets, preds, target_names=CFG.classes)

    tqdm.write('Val set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        val_loss, 100. * correct / len(val_loader.dataset)))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'val_loss': val_loss,
                   'val_accuracy': correct / len(val_loader.dataset)})

### Основная функция для прогона моделей

In [ ]:
def main_kolesa(model, train_df, val_df):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))
        # логируем архитектуру модели
        # https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})

    seed_everything(CFG.seed)
    # https://stackoverflow.com/questions/63423463/using-pytorch-cuda-on-macbook-pro
    # т.к. на macbook на их процессорах apple silicon нет cuda (только для карт nvidia), то используем альтеранативу, но если есть cuda - то используем ее
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    # наши картинки в исходном размере 750*470 + если домножить на число каналов (3), то получится около миллиарда параметра на вход.
    # Это довольно много так что пропорционально уменьшим размерность. У нас 750 к 470 это, можно сказать, 1.6 к 1. Уменьшим 750 до 160 => 470 -> 96.
    # будем использовать различные методы аугментации https://docs.pytorch.org/vision/0.13/transforms.html
    train_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    test_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    train_ds = KolesaDataset(train_df, transform=train_transform)
    val_ds = KolesaDataset(val_df, transform=test_transform)
    # test_ds = KolesaDataset(test_df, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG.train_batch_size, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG.test_batch_size, **kwargs)
    # test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    if CFG.wandb:
        wandb.watch(model, log='all')

    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val(model, device, val_loader, criterion, CFG.wandb)

    print('Training is ended!')

### Сохранение моделей

Чтобы у каждой модели сохранялся артефакт, добавим маленькую вспомогательную функцию. `main_kolesa` логирует loss/accuracy по эпохам в W&B сам, а эта функция дописывает в тот же запуск файл с весами.

In [ ]:
def save_model_artifact(model, name):
    # сохранение модели как артефакт
    # https://docs.wandb.ai/guides/artifacts
    path = base_dir / 'models' / f'{name}.pt' #путь к файлу в зависимости от имени модели (передаем в начале), .pt -расширение для объектов pytorch
    torch.save(model.state_dict(), path) #сохраняем параметры модели
    if CFG.wandb:
        wandb.run.name = name   # называем запуск как модель
        artifact = wandb.Artifact(name, type='model') #создаем артефакт-контейннер
        artifact.add_file(path) #добавляем в него файл с параметрами модели (брали выше)
        wandb.log_artifact(artifact) #загружаем артефакт и привящываем к текущему щапуску
        wandb.finish()  # закрываем текущий запуск
    print(f'[OK] модель сохранена: {path}')

In [ ]:
# данные эксперимента в W&B
# https://docs.wandb.ai/guides/artifacts
if CFG.wandb:
    wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, name='dataset')
    art = wandb.Artifact('kolesa-data', type='dataset')
    art.add_file('../data/prepared_df_images.csv')
    art.add_file('../data/brand_df_images.csv')
    art.add_dir('../data/images', name='images')
    wandb.log_artifact(art)
    wandb.finish()


Отдельно будем смотреть в этом ноутбуке AlexNet

## FineTuned AlexNet

Далее попробуем в качестве другого подхода взять уже ранее предобученную модель. Возьмем AlexNet

In [ ]:
from torchvision.models import alexnet
from torchvision.models import AlexNet_Weights

num_in_features = 9216 # Alex net после сверток выдает 256 каналов, а пуллинг сжимает картинку до 6 x 6
num_out_features = 2 # Sedan и SUV

CFG.lr = 1e-4 # learning rate понижаем для более корректного шага модели

#### Задача на тип кузова

In [ ]:
CFG.classes = all_body_types

model_alexnet_bt = alexnet(weights=AlexNet_Weights.DEFAULT) # DEFAULT - рекомендованный вариант
model_alexnet_bt.classifier = torch.nn.Linear(num_in_features, num_out_features) # alexnet.classifier по умолчанию выдает 1000 классов, а нам нужно 2, поэтому сокращаем

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 144MB/s]


In [ ]:
summary(model=model_alexnet_bt,
        input_size=(32, 3, 96, 160), #картинкики, канал, высота и ширина
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
) # прогоняем фиктивный батч через модель

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
AlexNet                                  [32, 3, 96, 160]     [32, 2]              --                   True
├─Sequential: 1-1                        [32, 3, 96, 160]     [32, 256, 2, 4]      --                   True
│    └─Conv2d: 2-1                       [32, 3, 96, 160]     [32, 64, 23, 39]     23,296               True
│    └─ReLU: 2-2                         [32, 64, 23, 39]     [32, 64, 23, 39]     --                   --
│    └─MaxPool2d: 2-3                    [32, 64, 23, 39]     [32, 64, 11, 19]     --                   --
│    └─Conv2d: 2-4                       [32, 64, 11, 19]     [32, 192, 11, 19]    307,392              True
│    └─ReLU: 2-5                         [32, 192, 11, 19]    [32, 192, 11, 19]    --                   --
│    └─MaxPool2d: 2-6                    [32, 192, 11, 19]    [32, 192, 5, 9]      --                   --
│    └─Conv2d: 2-7    

In [ ]:
main_kolesa(model_alexnet_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_alexnet_bt, 'AlexNet_finetune_body_type')

train_accuracy,▃▁▄▆▅▆█▇▇▇
train_loss,█▂▅▁▁▂▂▃▁▁
val_accuracy,▆▆▁▇▇▆█▃▄█
val_loss,▃▁▅▃▃▃▃▆█▅
train_accuracy,0.98743
train_loss,0.0074
val_accuracy,0.80752
val_loss,0.7562



Epoch: 1


100%|██████████| 97/97 [00:36<00:00,  2.62it/s]



Train set: Average loss: 0.0444, Accuracy: 98.61%
Val set: Average loss: 0.9117, Accuracy: 81.65%

Classification report:
              precision    recall  f1-score   support

       sedan       0.81      0.93      0.86       418
         SUV       0.83      0.63      0.72       247

    accuracy                           0.82       665
   macro avg       0.82      0.78      0.79       665
weighted avg       0.82      0.82      0.81       665


Epoch: 2


100%|██████████| 97/97 [00:39<00:00,  2.47it/s]


...


Train set: Average loss: 0.2770, Accuracy: 98.55%
Val set: Average loss: 0.8335, Accuracy: 81.35%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.89      0.86       418
         SUV       0.79      0.68      0.73       247

    accuracy                           0.81       665
   macro avg       0.81      0.79      0.79       665
weighted avg       0.81      0.81      0.81       665


Epoch: 10


100%|██████████| 97/97 [00:37<00:00,  2.60it/s]



Train set: Average loss: 0.0086, Accuracy: 98.49%
Val set: Average loss: 0.7432, Accuracy: 80.75%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.87      0.85       418
         SUV       0.76      0.70      0.73       247

    accuracy                           0.81       665
   macro avg       0.80      0.79      0.79       665
weighted avg       0.81      0.81      0.81       665

Training is ended!


train_accuracy,▇▅▆▆▁▇▇█▆▆
train_loss,▂▂▂▃▅▁▃▂█▁
val_accuracy,▇▁▆██▇▇▆▇▇
val_loss,▄█▁▃▂▄▁▃▃▁
train_accuracy,0.98485
train_loss,0.00856
val_accuracy,0.80752
val_loss,0.74315


[OK] модель сохранена: /content/data/models/AlexNet_finetune_body_type.pt


Неплохой accuracy - 82.86% на лучшей эпохе, на 10 п.п. лучше чем было на нашем CNN. Но есть куда расти, recall у SUV сильно страдает.

#### Задача на определение марки

In [ ]:
num_out_features = 8 # 8 разных марок
CFG.classes = all_brands

model_alexnet_brand = alexnet(weights=AlexNet_Weights.DEFAULT) # DEFAULT - рекомендованный вариант
model_alexnet_brand.classifier = torch.nn.Linear(num_in_features, num_out_features) # alexnet.classifier по умолчанию выдает 1000 классов, а нам нужно 8, поэтому сокращаем

In [ ]:
summary(model=model_alexnet_brand,
        input_size=(32, 3, 96, 160), # картинкики, канал, высота и ширина
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
) # прогоняем фиктивный батч через модель

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
AlexNet                                  [32, 3, 96, 160]     [32, 8]              --                   True
├─Sequential: 1-1                        [32, 3, 96, 160]     [32, 256, 2, 4]      --                   True
│    └─Conv2d: 2-1                       [32, 3, 96, 160]     [32, 64, 23, 39]     23,296               True
│    └─ReLU: 2-2                         [32, 64, 23, 39]     [32, 64, 23, 39]     --                   --
│    └─MaxPool2d: 2-3                    [32, 64, 23, 39]     [32, 64, 11, 19]     --                   --
│    └─Conv2d: 2-4                       [32, 64, 11, 19]     [32, 192, 11, 19]    307,392              True
│    └─ReLU: 2-5                         [32, 192, 11, 19]    [32, 192, 11, 19]    --                   --
│    └─MaxPool2d: 2-6                    [32, 192, 11, 19]    [32, 192, 5, 9]      --                   --
│    └─Conv2d: 2-7    

In [ ]:
main_kolesa(model_alexnet_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_alexnet_brand, 'AlexNet_finetune_brand')


Epoch: 1


100%|██████████| 97/97 [00:37<00:00,  2.57it/s]



Train set: Average loss: 2.0299, Accuracy: 16.79%
Val set: Average loss: 2.0817, Accuracy: 18.95%

Classification report:
               precision    recall  f1-score   support

         Audi       0.23      0.47      0.31        73
          BMW       0.14      0.12      0.13        90
    Chevrolet       0.25      0.37      0.30        79
      Hyundai       0.17      0.01      0.02        85
          Kia       0.28      0.22      0.25        77
        Lexus       0.14      0.08      0.10        95
Mercedes-Benz       0.11      0.17      0.14        81
       Toyota       0.16      0.14      0.15        85

     accuracy                           0.19       665
    macro avg       0.19      0.20      0.17       665
 weighted avg       0.18      0.19      0.17       665


Epoch: 2


100%|██████████| 97/97 [00:37<00:00,  2.57it/s]



Train set: Average loss: 1.9008, Accuracy: 23.98%
Val set: Average loss: 2.0287, Accuracy: 21.50%

Classification report:
               precision    recall  f1-score   support

         Audi       0.26      0.60      0.36        73
          BMW       0.19      0.23      0.21        90
    Chevrolet       0.37      0.22      0.27        79
      Hyundai       0.25      0.06      0.10        85
          Kia       0.34      0.22      0.27        77
        Lexus       0.17      0.15      0.16        95
Mercedes-Benz       0.16      0.23      0.19        81
       Toyota       0.10      0.07      0.08        85

     accuracy                           0.22       665
    macro avg       0.23      0.22      0.20       665
 weighted avg       0.22      0.22      0.20       665


Epoch: 3


...


Train set: Average loss: 1.4461, Accuracy: 60.55%
Val set: Average loss: 2.0308, Accuracy: 34.29%

Classification report:
               precision    recall  f1-score   support

         Audi       0.67      0.19      0.30        73
          BMW       0.54      0.44      0.49        90
    Chevrolet       0.46      0.27      0.34        79
      Hyundai       0.36      0.33      0.35        85
          Kia       0.35      0.40      0.37        77
        Lexus       0.27      0.37      0.31        95
Mercedes-Benz       0.26      0.48      0.34        81
       Toyota       0.24      0.24      0.24        85

     accuracy                           0.34       665
    macro avg       0.39      0.34      0.34       665
 weighted avg       0.39      0.34      0.34       665


Epoch: 10


100%|██████████| 97/97 [00:37<00:00,  2.57it/s]



Train set: Average loss: 1.1504, Accuracy: 64.20%
Val set: Average loss: 2.2243, Accuracy: 32.18%

Classification report:
               precision    recall  f1-score   support

         Audi       0.62      0.21      0.31        73
          BMW       0.40      0.58      0.47        90
    Chevrolet       0.57      0.15      0.24        79
      Hyundai       0.34      0.41      0.37        85
          Kia       0.39      0.26      0.31        77
        Lexus       0.29      0.32      0.30        95
Mercedes-Benz       0.24      0.32      0.28        81
       Toyota       0.20      0.28      0.23        85

     accuracy                           0.32       665
    macro avg       0.38      0.32      0.31       665
 weighted avg       0.37      0.32      0.32       665

Training is ended!


train_accuracy,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▇▆▆▅▃▂▃▁
val_accuracy,▁▂▃▅▅▆▇▇█▇
val_loss,▅▄▂▂▁▂▂▃▄█
train_accuracy,0.64196
train_loss,1.15044
val_accuracy,0.3218
val_loss,2.22431


[OK] модель сохранена: /content/data/models/AlexNet_finetune_brand.pt


34.29% на лучшей эпохе - все также плохо, при этом не сильно далеко ушли от CNN.

Попробуем второй вариант (который был на семинаре) дообучать только верхние слои + классификатор.

Замороженные параметры (`requires_grad=False`) оптимизатор просто не обновляет.




#### Задача на тип кузова

In [ ]:
CFG.classes = all_body_types
num_out_features = 2

model_alexnet_frozen_bt = alexnet(weights=AlexNet_Weights.DEFAULT)

# замораживаем первые 6 параметров f
layers_to_freeze = 6
for i, (name, param) in enumerate(model_alexnet_frozen_bt.features.named_parameters()):
    if i < layers_to_freeze:
        param.requires_grad = False
    print(f'{name:30}{param.requires_grad}') # выводим имя параметра и его статус (будет использоваться или гет)
    # name:30 выравниваение параметров по ширине в 30 символов (чтобы аккуратно было)

model_alexnet_frozen_bt.classifier = torch.nn.Linear(num_in_features, num_out_features)

0.weight                      False
0.bias                        False
3.weight                      False
3.bias                        False
6.weight                      False
6.bias                        False
8.weight                      True
8.bias                        True
10.weight                     True
10.bias                       True


In [ ]:
main_kolesa(model_alexnet_frozen_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_alexnet_frozen_bt, 'AlexNet_finetune_frozen_body_type')


Epoch: 1


100%|██████████| 97/97 [00:36<00:00,  2.66it/s]



Train set: Average loss: 0.5590, Accuracy: 67.13%
Val set: Average loss: 0.5023, Accuracy: 70.98%

Classification report:
              precision    recall  f1-score   support

       sedan       0.81      0.70      0.75       418
         SUV       0.59      0.72      0.65       247

    accuracy                           0.71       665
   macro avg       0.70      0.71      0.70       665
weighted avg       0.73      0.71      0.71       665


Epoch: 2


100%|██████████| 97/97 [00:36<00:00,  2.65it/s]



Train set: Average loss: 0.5024, Accuracy: 74.38%
Val set: Average loss: 0.4957, Accuracy: 70.23%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.66      0.74       418
         SUV       0.57      0.77      0.66       247

    accuracy                           0.70       665
   macro avg       0.70      0.72      0.70       665
weighted avg       0.74      0.70      0.71       665


Epoch: 3


...


Train set: Average loss: 0.2552, Accuracy: 89.24%
Val set: Average loss: 0.4210, Accuracy: 77.14%

Classification report:
              precision    recall  f1-score   support

       sedan       0.84      0.79      0.81       418
         SUV       0.68      0.74      0.71       247

    accuracy                           0.77       665
   macro avg       0.76      0.77      0.76       665
weighted avg       0.78      0.77      0.77       665


Epoch: 10


100%|██████████| 97/97 [00:37<00:00,  2.59it/s]



Train set: Average loss: 0.1094, Accuracy: 92.36%
Val set: Average loss: 0.4348, Accuracy: 76.99%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.77      0.81       418
         SUV       0.66      0.77      0.71       247

    accuracy                           0.77       665
   macro avg       0.76      0.77      0.76       665
weighted avg       0.78      0.77      0.77       665

Training is ended!


train_accuracy,▁▃▄▅▅▆▆▇▇█
train_loss,▇▆█▅▆▄▃▅▃▁
val_accuracy,▂▁▅▅▅▅▆█▆▆
val_loss,██▃▃▁▃▁▃▄▄
train_accuracy,0.92362
train_loss,0.10937
val_accuracy,0.76992
val_loss,0.43476


[OK] модель сохранена: /content/data/models/AlexNet_finetune_frozen_body_type.pt


Accuracy 80.00% - вышло даже хуже чем без заморозки. Но зато recall сравнялся с precision. То есть модель хуже в целом по точности, но стабильнее по precision и recall

#### Задача на определение марки

In [ ]:
CFG.classes = all_brands
num_out_features = 8

model_alexnet_frozen_brand = alexnet(weights=AlexNet_Weights.DEFAULT)

# замораживаем первые 6 параметров f
layers_to_freeze = 6
for i, (name, param) in enumerate(model_alexnet_frozen_brand.features.named_parameters()):
    if i < layers_to_freeze:
        param.requires_grad = False
    print(f'{name:30}{param.requires_grad}') # выводим имя параметра и его статус (будет использоваться или гет)
    # name:30 выравниваение параметров по ширине в 30 символов (чтобы аккуратно было)

model_alexnet_frozen_brand.classifier = torch.nn.Linear(num_in_features, num_out_features)

0.weight                      False
0.bias                        False
3.weight                      False
3.bias                        False
6.weight                      False
6.bias                        False
8.weight                      True
8.bias                        True
10.weight                     True
10.bias                       True


In [ ]:
main_kolesa(model_alexnet_frozen_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_alexnet_frozen_brand, 'AlexNet_finetune_frozen_brand')


Epoch: 1


100%|██████████| 97/97 [00:36<00:00,  2.63it/s]



Train set: Average loss: 1.9740, Accuracy: 17.56%
Val set: Average loss: 2.0996, Accuracy: 19.25%

Classification report:
               precision    recall  f1-score   support

         Audi       0.22      0.33      0.27        73
          BMW       0.18      0.12      0.15        90
    Chevrolet       0.18      0.46      0.26        79
      Hyundai       0.13      0.04      0.06        85
          Kia       0.39      0.21      0.27        77
        Lexus       0.18      0.15      0.16        95
Mercedes-Benz       0.15      0.23      0.18        81
       Toyota       0.17      0.06      0.09        85

     accuracy                           0.19       665
    macro avg       0.20      0.20      0.18       665
 weighted avg       0.20      0.19      0.17       665


Epoch: 2


100%|██████████| 97/97 [00:37<00:00,  2.61it/s]



Train set: Average loss: 1.8352, Accuracy: 27.68%
Val set: Average loss: 2.0022, Accuracy: 24.66%

Classification report:
               precision    recall  f1-score   support

         Audi       0.28      0.51      0.36        73
          BMW       0.24      0.24      0.24        90
    Chevrolet       0.35      0.34      0.34        79
      Hyundai       0.20      0.07      0.10        85
          Kia       0.41      0.25      0.31        77
        Lexus       0.22      0.29      0.25        95
Mercedes-Benz       0.16      0.27      0.20        81
       Toyota       0.12      0.04      0.05        85

     accuracy                           0.25       665
    macro avg       0.25      0.25      0.23       665
 weighted avg       0.24      0.25      0.23       665


Epoch: 3


...


Train set: Average loss: 1.0241, Accuracy: 64.58%
Val set: Average loss: 2.1830, Accuracy: 32.78%

Classification report:
               precision    recall  f1-score   support

         Audi       0.36      0.37      0.36        73
          BMW       0.45      0.48      0.46        90
    Chevrolet       0.53      0.24      0.33        79
      Hyundai       0.39      0.16      0.23        85
          Kia       0.33      0.34      0.33        77
        Lexus       0.32      0.36      0.34        95
Mercedes-Benz       0.24      0.40      0.29        81
       Toyota       0.23      0.27      0.25        85

     accuracy                           0.33       665
    macro avg       0.35      0.33      0.33       665
 weighted avg       0.35      0.33      0.33       665


Epoch: 10


100%|██████████| 97/97 [00:37<00:00,  2.61it/s]



Train set: Average loss: 0.7719, Accuracy: 68.77%
Val set: Average loss: 2.2851, Accuracy: 33.53%

Classification report:
               precision    recall  f1-score   support

         Audi       0.41      0.45      0.43        73
          BMW       0.46      0.48      0.47        90
    Chevrolet       0.42      0.19      0.26        79
      Hyundai       0.35      0.28      0.31        85
          Kia       0.43      0.21      0.28        77
        Lexus       0.28      0.34      0.30        95
Mercedes-Benz       0.26      0.43      0.32        81
       Toyota       0.26      0.29      0.27        85

     accuracy                           0.34       665
    macro avg       0.36      0.33      0.33       665
 weighted avg       0.35      0.34      0.33       665

Training is ended!


train_accuracy,▁▂▃▄▅▆▆▇▇█
train_loss,█▇▅▆▄▅▃▃▂▁
val_accuracy,▁▄▅▆▆▇▇▇██
val_loss,▄▂▁▂▁▃▄▄▆█
train_accuracy,0.68772
train_loss,0.7719
val_accuracy,0.33534
val_loss,2.28507


[OK] модель сохранена: /content/data/models/AlexNet_finetune_frozen_brand.pt


33.53% - тоже хуже, но местами тоже стабильнее. К примеру Audi на прошлой модели показал precision 67% и recall 19% - прям большая разница. А здесь же recall даже чуть выше - 45% и 41% у precision. Так что вывод тот же - просто стабильнее, но сама по себе все еще слабовата.